In [ ]:
import os

# ============================================================
# ENVIRONMENT SETUP — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab ───────────────────────────────────
# Mounts Drive and changes to your project folder so all
# outputs (plots, pkl) save there automatically.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/ML-Assignment/attemp2'
os.chdir(DRIVE_PATH)
print(f"✓ Working directory: {DRIVE_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the two lines below.
# Make sure cardio_train.csv is in the same folder as this notebook.
# LOCAL_PATH = os.path.dirname(os.path.abspath("__file__"))
# os.chdir(LOCAL_PATH)

# Create output directories
os.makedirs("plots",  exist_ok=True)
os.makedirs("report", exist_ok=True)
print("✓ Output folders ready.")


In [ ]:
# ============================================================
# DATASET LOADING — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab — upload file via dialog ─────────
from google.colab import files
print("Click Choose Files and select cardio_train.csv")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"✓ Uploaded: {DATASET_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the line below.
# DATASET_PATH = "cardio_train.csv"  # must be in the same folder as this notebook
# print(f"✓ Dataset path: {DATASET_PATH}")


# Cardiovascular Disease Prediction
## Notebook 01: Exploratory Data Analysis & Data Preprocessing
---
**Author:** MS26912448 (Member 1)
**Dataset:** Cardiovascular Disease Dataset — Kaggle

### Objectives
This notebook covers:
1. Loading and inspecting the raw dataset
2. Exploratory Data Analysis (EDA) with visualisations
3. Data Preprocessing pipeline
4. Saving clean, scaled data for model training

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Create output directories
os.makedirs('plots', exist_ok=True)

print("✓ All libraries imported successfully.")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  sklearn : ", end='')
import sklearn; print(sklearn.__version__)

## 1. Load Dataset

The dataset is stored as a semicolon-delimited CSV. We load it using `pd.read_csv` with `sep=';'`.

In [ ]:
# Load the cardiovascular disease dataset
# NOTE: The dataset uses semicolon (;) as the delimiter
df = pd.read_csv(DATASET_PATH, sep=';')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

## 2. Initial Data Inspection

In [ ]:
# Display column data types and non-null counts
print("=== Dataset Information ===")
df.info()

In [ ]:
# Descriptive statistics for all columns
print("=== Descriptive Statistics ===")
df.describe().round(2)

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Check data types
print("\nColumn data types:")
print(df.dtypes)

## 3. Missing Value Analysis

In [ ]:
# Check for missing values in each column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("=== Missing Values Summary ===")
print(missing_df[missing_df['Missing Count'] > 0] if missing.any()
      else "No missing values found in the dataset.")

## 4. Target Variable Analysis

The target variable `cardio` is binary:
- **0**: No cardiovascular disease
- **1**: Has cardiovascular disease

We first check for class imbalance, which affects model selection and evaluation.

In [ ]:
# Visualise target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['cardio'].value_counts().sort_index()
axes[0].bar(['No Disease (0)', 'Has Disease (1)'], counts.values,
            color=['#2196F3', '#F44336'], alpha=0.85, edgecolor='black', linewidth=0.8)
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['No Disease', 'Has Disease'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, explode=(0.05, 0))
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Cardiovascular Disease – Target Variable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Class 0 (No Disease): {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")
print(f"Class 1 (Has Disease): {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")
print(f"\nDataset is approximately balanced — no SMOTE required.")

## 5. Feature Distributions

### 5.1 Age Distribution
Age is stored in **days**. We convert to years for readability.

In [ ]:
# Age distribution (converting from days to years for visualisation)
age_years = df['age'] / 365

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(age_years, bins=35, color='#42A5F5', edgecolor='black', alpha=0.8, linewidth=0.6)
axes[0].set_title('Age Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(age_years.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {age_years.mean():.1f} yrs')
axes[0].legend()

axes[1].boxplot(age_years, vert=True, patch_artist=True,
               boxprops=dict(facecolor='#42A5F5', alpha=0.8))
axes[1].set_title('Age Boxplot', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Age (years)')

plt.suptitle('Age Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Age range : {age_years.min():.0f} – {age_years.max():.0f} years")
print(f"Mean age  : {age_years.mean():.1f} years")
print(f"Std dev   : {age_years.std():.1f} years")

### 5.2 Numerical Features (Height, Weight, Blood Pressure)

In [ ]:
# Distribution of numerical features with boxplots
num_cols = ['height', 'weight', 'ap_hi', 'ap_lo']
colors   = ['#66BB6A', '#FFA726', '#EF5350', '#AB47BC']
labels   = ['Height (cm)', 'Weight (kg)', 'Systolic BP', 'Diastolic BP']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, (col, clr, lbl) in enumerate(zip(num_cols, colors, labels)):
    # Histogram
    axes[0, i].hist(df[col], bins=40, color=clr, edgecolor='black', alpha=0.8, linewidth=0.5)
    axes[0, i].set_title(lbl, fontsize=11, fontweight='bold')
    axes[0, i].set_ylabel('Frequency')
    # Boxplot
    axes[1, i].boxplot(df[col], vert=True, patch_artist=True,
                       boxprops=dict(facecolor=clr, alpha=0.8))
    axes[1, i].set_title(f'{lbl} Boxplot', fontsize=11)

plt.suptitle('Numerical Feature Distributions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/03_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Note significant outliers in blood pressure
print("⚠ Note: Blood pressure columns contain unrealistic outliers.")
print(f"  ap_hi max: {df['ap_hi'].max()} | ap_lo min: {df['ap_lo'].min()}")
print("  These will be removed in the preprocessing step.")

### 5.3 Categorical Features